# BƯỚC 4: LỰA CHỌN VÀ TỐI ƯU MÔ HÌNH (MODEL SELECTION & TUNING)

**Mục tiêu:** Xây dựng và so sánh 3 mô hình Học máy để giải quyết bài toán Phân loại (Dự đoán xu hướng Giá Tăng/Giảm).

**Yêu cầu theo barem:**
1. **Thuật toán áp dụng:** Logistic Regression, Random Forest, XGBoost.
2. **Tối ưu siêu tham số (Hyperparameter Tuning):** Dò tìm bằng lưới (Grid Search) đúng các tham số barem chỉ định.
3. **Chống rò rỉ dữ liệu:** Bắt buộc sử dụng `TimeSeriesSplit` trong quá trình Cross-Validation 

[Image of Cross validation in time series]
.
4. **Báo cáo phân tích:** Vẽ biểu đồ đánh giá tác động của `learning_rate` (XGBoost) đến quá trình hội tụ.

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print("--- 4: Nạp dữ liệu và Chuẩn bị Mô hình ---")

target_tickers = ['VNM', 'FPT', 'VIC']
processed_dir = '../data/processed'
data_frames = {}

for ticker in target_tickers:
    file_path = os.path.join(processed_dir, f"{ticker}_features.csv")
    
    #Thoát nếu thiếu file
    if not os.path.exists(file_path):
        print(f"[LỖI] Không tìm thấy file dữ liệu: {file_path}")
        continue
        
    df = pd.read_csv(file_path, index_col='Date', parse_dates=True)
    data_frames[ticker] = df
    
    print(f"-> Đã nạp {ticker} ({df.shape[0]} dòng, {df.shape[1]} cột)")

## 4.1. Phân chia tập huấn luyện và tập kiểm thử (Train/Test Split)

Tuân thủ nguyên tắc chuỗi thời gian:
- **Tập Train:** Dữ liệu quá khứ (Từ 2020 đến hết 31/12/2024).
- **Tập Test:** Dữ liệu tương lai (Từ 01/01/2025 trở đi).
- **Biến mục tiêu (Target):** `Target_Class` (1 = Tăng, 0 = Giảm/Đi ngang).

In [ ]:
print("--- 4.1: Chia tập Train/Test theo mốc thời gian ---")

split_date = '2024-12-31'
features_dict = {}

for ticker, df in data_frames.items():
    #Tách Features (X) và Target (y)
    X = df.drop(columns=['Target_Class'])
    y = df['Target_Class'] 
    
    #Chia tập Train/Test theo mốc thời gian
    train_mask = X.index <= split_date
    test_mask = X.index > split_date
    
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]
    
    features_dict[ticker] = {
        'X_train': X_train, 'y_train': y_train,
        'X_test': X_test, 'y_test': y_test
    }
    
    #In ra số lượng dòng thực tế của Train/Test sau khi chia
    print(f"-> {ticker}: Train ({len(X_train)} dòng) | Test ({len(X_test)} dòng)")

## 4.2. Tìm kiếm Siêu tham số (GridSearchCV) kết hợp TimeSeriesSplit

Không gian Lưới tham số (Param Grid) được thiết lập bám sát 100% yêu cầu đồ án:
- **Logistic Regression**: `C`, `max_iter`
- **Random Forest**: `n_estimators`, `max_depth`, `min_samples_split`
- **XGBoost**: `n_estimators`, `learning_rate`, `max_depth`, `subsample`


In [ ]:
print("--- 4.2: Huấn luyện và Tối ưu Mô hình (Grid Search) ---")

model_dir = '../models'
os.makedirs(model_dir, exist_ok=True)

tscv = TimeSeriesSplit(n_splits=5) 

models_and_params = {
    'Logistic Regression': {
        'model': LogisticRegression(class_weight='balanced', random_state=42),
        'params': {'C': [0.1, 1, 10], 'max_iter': [100, 500]}
    },
    'Random Forest': {
        'model': RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5, 10]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': [100, 200], 
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'max_depth': [3, 5, 7],
            'subsample': [0.8, 1.0],
            'colsample_bytree': [0.8, 1.0]
        }
    }
}

all_results = []
best_models_dict = {}
xgb_cv_results = {} 

for ticker, data in features_dict.items():
    print(f"\n[Tuning Hyperparameters - {ticker}]")
    
    X_train, y_train = data['X_train'], data['y_train']
    X_test, y_test = data['X_test'], data['y_test']
    
    #Tính toán tỷ lệ chênh lệch nhãn (0/1) cho XGBoost
    num_neg = np.sum(y_train == 0)
    num_pos = np.sum(y_train == 1)
    ratio = float(num_neg) / num_pos if num_pos > 0 else 1
    models_and_params['XGBoost']['model'].set_params(scale_pos_weight=ratio)
    
    best_models_dict[ticker] = {}
    
    for model_name, mp in models_and_params.items():
        print(f" -> Đang train {model_name}...")
        
        grid_search = GridSearchCV(
            estimator=mp['model'],
            param_grid=mp['params'],
            cv=tscv,
            scoring='f1_macro', 
            n_jobs=-1
        )
        grid_search.fit(X_train, y_train)
        
        #Lưu log CV của XGBoost để phân tích hội tụ sau này
        if model_name == 'XGBoost':
            xgb_cv_results[ticker] = grid_search.cv_results_
            
        best_model = grid_search.best_estimator_
        best_models_dict[ticker][model_name] = best_model
        
        #Lưu file model
        safe_model_name = model_name.replace(" ", "_").lower()
        model_path = os.path.join(model_dir, f"{safe_model_name}_{ticker}.pkl")
        with open(model_path, 'wb') as f:
            pickle.dump(best_model, f)
            
        #Đánh giá model trên tập Test
        y_pred = best_model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')
        
        all_results.append({
            'Mã CP': ticker,
            'Thuật toán': model_name,
            'Accuracy (%)': round(acc * 100, 2),
            'F1-Macro (%)': round(f1 * 100, 2),
            'Best Params': str(grid_search.best_params_)
        })

print("\n--- BẢNG TỔNG HỢP HIỆU NĂNG MÔ HÌNH ---")
results_df = pd.DataFrame(all_results)
for ticker in target_tickers:
    print(f"\n=> Báo cáo cho {ticker}:")
    df_ticker = results_df[results_df['Mã CP'] == ticker].sort_values(by='F1-Macro (%)', ascending=False)
    display(df_ticker[['Thuật toán', 'Accuracy (%)', 'F1-Macro (%)', 'Best Params']].reset_index(drop=True))

## 4.3. Báo cáo ảnh hưởng của Learning Rate đến quá trình hội tụ (XGBoost)
Trích xuất dữ liệu từ lưới GridSearchCV của XGBoost để vẽ biểu đồ chứng minh tham số `learning_rate` định hình quá trình học của mô hình như thế nào.

In [ ]:
print("--- 4.3: Phân tích độ nhạy của Learning Rate (XGBoost) ---")

output_dir = '../outputs'
os.makedirs(output_dir, exist_ok=True)

for ticker in target_tickers:
    if ticker not in xgb_cv_results:
        print(f"[{ticker}] Không có log lịch sử của XGBoost, bỏ qua.")
        continue
        
    cv_results = xgb_cv_results[ticker]
    
    #Ép kiểu list()
    lr_df = pd.DataFrame({
        'Learning_Rate': list(cv_results['param_learning_rate']),
        'F1_Score': cv_results['mean_test_score']
    })
    
    #Gom nhóm tính trung bình F1 cho từng mốc Learning Rate
    lr_summary = lr_df.groupby('Learning_Rate')['F1_Score'].mean().reset_index()
    
    plt.figure(figsize=(8, 5))
    sns.lineplot(data=lr_summary, x='Learning_Rate', y='F1_Score', marker='o', linewidth=2)
    
    plt.title(f'Tác động của Learning Rate (XGBoost) - {ticker}')
    plt.xlabel('Learning Rate')
    plt.ylabel('Mean CV F1-Score')
    plt.xticks(lr_summary['Learning_Rate'].unique())
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    #Lưu
    save_path = os.path.join(output_dir, f'04_learning_rate_impact_{ticker}.png')
    plt.savefig(save_path, dpi=300)
    plt.show()
    
    print(f"-> Đã xuất biểu đồ LR cho {ticker}")